In [1]:
!pip install FastAPI
from fastapi import FastAPI
import pickle

In [3]:
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

class PostRecommender:
    def __init__(self, post_vectors, posts_df, interaction_matrix, user_factors, users_df):
        self.post_vectors = post_vectors
        self.posts_df = posts_df
        self.interaction_matrix = interaction_matrix
        self.user_factors = user_factors
        self.users_df = users_df

    def get_user_content_profile(self, user_id):
        user_views = views_exploded[views_exploded['userId'] == user_id]
        viewed_post_ids = user_views['tagId'].tolist()
        weights = user_views['weight'].values

        viewed_vectors = self.post_vectors[posts_df['_id'].isin(viewed_post_ids)]
        
        if viewed_vectors.shape[0] == 0:
            return np.zeros((1, self.post_vectors.shape[1]))

        if len(weights) == 0:
            return viewed_vectors.mean(axis=0)

        weighted_vectors = viewed_vectors.multiply(weights, axis=0)
        return weighted_vectors.sum(axis=0) / weights.sum()

    def recommend_posts(self, user_id, top_n=5):
        user_profile = self.get_user_content_profile(user_id)
        content_scores = cosine_similarity(user_profile, self.post_vectors).flatten()

        post_ids = self.posts_df['_id'].tolist()
        collab_scores = np.zeros(len(post_ids))

        if user_id in self.interaction_matrix.index:
            user_index = self.interaction_matrix.index.get_loc(user_id)
            user_collab_scores = self.user_factors[user_index]
            collab_score_dict = dict(zip(self.interaction_matrix.columns, user_collab_scores))
            collab_scores = np.array([collab_score_dict.get(post_id, 0) for post_id in post_ids])

        popularity_scores = self.posts_df['popularity_score'].values

        # Normalize scores
        content_scores /= content_scores.max() if content_scores.max() > 0 else 1
        collab_scores /= collab_scores.max() if collab_scores.max() > 0 else 1
        popularity_scores /= popularity_scores.max() if popularity_scores.max() > 0 else 1

        # Get age group weight
        user_age_group = self.users_df.loc[self.users_df['_id'] == user_id, 'age_group'].values[0] if user_id in self.users_df['_id'].values else 'unknown'
        age_weight = {'teen': 1.0, 'young_adult': 1.2, 'adult': 1.1, 'senior': 0.9, 'unknown': 1.0}
        age_group_weight = age_weight.get(user_age_group, 1.0)

        # Combine scores with randomness
        combined_scores = (
            0.4 * content_scores +
            0.3 * collab_scores +
            0.25 * popularity_scores +
            0.03 * age_group_weight +
            np.random.normal(0, 0.02, len(content_scores))
        )

        # Exclude user's own posts
        user_posts = self.posts_df[self.posts_df['userId'] == user_id]['_id'].tolist()
        recommended_indices = [
            i for i in np.argsort(combined_scores)[-top_n * 2:][::-1]
            if post_ids[i] not in user_posts
        ][:top_n]

        return self.posts_df.iloc[recommended_indices][['_id', 'postMessage', 'popularity_score']]

In [5]:
with open("Downloads/post-recommendation-api/champhunt_pitch.pkl", "rb") as file:
    recommender = pickle.load(file)

print("Model loaded successfully!")

Model loaded successfully!


In [18]:
print("Available attributes in model:", dir(recommender))

Available attributes in model: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'get_user_content_profile', 'interaction_matrix', 'post_vectors', 'posts_df', 'recommend_posts', 'user_factors', 'users_df', 'views_exploded']


In [21]:
print(recommender.views_exploded.head())

                        _id                    userId  \
0  679b4e9b74c475998774e340  6246842ba1185e0d52b14468   
0  679b4e9b74c475998774e340  6246842ba1185e0d52b14468   
0  679b4e9b74c475998774e340  6246842ba1185e0d52b14468   
0  679b4e9b74c475998774e340  6246842ba1185e0d52b14468   
0  679b4e9b74c475998774e340  6246842ba1185e0d52b14468   

                                          location               createdAt  \
0  {'latitude': 0, 'longitude': 0, 'source': 'ip'} 2025-01-30 10:04:11.110   
0  {'latitude': 0, 'longitude': 0, 'source': 'ip'} 2025-01-30 10:04:11.110   
0  {'latitude': 0, 'longitude': 0, 'source': 'ip'} 2025-01-30 10:04:11.110   
0  {'latitude': 0, 'longitude': 0, 'source': 'ip'} 2025-01-30 10:04:11.110   
0  {'latitude': 0, 'longitude': 0, 'source': 'ip'} 2025-01-30 10:04:11.110   

                updatedAt  __v                     tagId  weight  \
0 2025-01-30 10:36:05.473  157  679b4e9b74c475998774e33d      16   
0 2025-01-30 10:36:05.473  157  679b4e9b74c475998774

In [7]:
!pip install fastapi uvicorn nest_asyncio scikit-learn numpy pandas

In [29]:
from fastapi import FastAPI
import pickle
import nest_asyncio
import uvicorn

post_vectors = recommender.post_vectors
posts_df = recommender.posts_df
interaction_matrix = recommender.interaction_matrix
user_factors = recommender.user_factors
users_df = recommender.users_df
views_exploded = recommender.views_exploded

# Initialize FastAPI app
app = FastAPI()

@app.get("/")
def home():
    return {"message": "Recommendation API is running!"}

@app.get("/recommend")
def get_recommendations(user_id: str, top_n: int = 10):
    """
    Get recommended posts for a given user ID.
    """
    try:
        recommendations = recommender.recommend_posts(user_id, top_n)
        recommendations['_id'] = recommendations['_id'].astype(str)
        return recommendations.to_dict(orient="records")  # Convert DataFrame to JSON
    except Exception as e:
        return {"error": str(e)}

# Fix async loop issues in Jupyter Notebook
nest_asyncio.apply()

# Run FastAPI server inside Jupyter
uvicorn.run(app, host="0.0.0.0", port=8000)


INFO:     Started server process [12032]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58508 - "GET /recommend?user_id=6246842ba1185e0d52b14468&top_n=5 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58509 - "GET /recommend?user_id=6246842ba1185e0d52b14468&top_n=10 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58510 - "GET /recommend?user_id=6246842ba1185e0d52b14468&top_n=20 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [12032]
